## Changelog
- parent: 20260507_214919_cd6c065b
- change: add GaragePLSTransformer(n_components=3, drop_originals=True) after
    AmesEncoder, completing the re-introduction of yesterday's FE stack on top
    of the Ames-aware preprocessing baseline.
- hypothesis: garage-PLS was the largest single LB lift in run 10 (-473) but
    showed CV-LB asymmetry (CV moved only -0.003 log-RMSE while LB moved much
    more). Adding it isolated on top of the FE stack lets us measure its
    contribution against the strongest baseline so far (run 14, public 14384.54).
    Worst case: lose the asymmetry and CV/LB regress in lockstep. Best case:
    the asymmetry replicates and we get another large LB drop.

In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")

In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

# Drop Id (row identifier, no signal) and SalePrice (target). Everything else
# goes through preprocessing.
DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

# Auto-detect num/cat from train; apply the same split to test.
NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

from utils.ames_sklearn_pipeline import (
    AmesNAImputer, AmesEncoder, TargetEncodeColumn, GaragePLSTransformer,
)
from utils.ames_feature_engineering import add_size_features, add_temporal_features

pipe = Pipeline([
    ("na",         AmesNAImputer()),         # absent-vs-missing aware NA handling
    ("size",       FunctionTransformer(
                       add_size_features,
                       kw_args={"drop_originals": True},
                   )),                       # TotalSF replaces TotalBsmtSF + 1stFlrSF + 2ndFlrSF
    ("fe",         FunctionTransformer(
                       add_temporal_features,
                       kw_args={"drop_originals": True},
                   )),                       # HouseAge etc. replace YearBuilt/YearRemodAdd/YrSold
    ("nbhd_te",    TargetEncodeColumn("Neighborhood")),  # 25 levels -> one float, CV-encoded
    ("encoder",    AmesEncoder()),           # quality ordinals + CentralAir + one-hot the rest
    ("garage_pls", GaragePLSTransformer(n_components=3, drop_originals=True)),  # 23-col garage block -> 3 supervised components
    ("model",      GradientBoostingRegressor(n_estimators=300, random_state=42)),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rmse = -scores
print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"Per-fold:            {np.round(rmse, 4).tolist()}")

pipe.fit(X, y)

In [ ]:
test_pred = np.expm1(pipe.predict(X_test))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)